# Module 3.2a — Structured / Constrained Decoding for Extraction
**DeskMate SLM Curriculum · Phase 3 · Notebook 17**

Run on **Colab T4** (recommended) or CPU (slower, falls back to smaller model).

By the end you will have:
- A JSON-schema-constrained extractor that *cannot* produce malformed output
- A side-by-side benchmark vs the Module 2.5 NER head
- `reports/constrained_decoding_benchmark.md`

Read `3.2a_constrained_decoding.md` for full theory.

---

## Step 0 — Install & Seed

In [ ]:
%%capture
!pip install -q outlines==0.0.46 transformers==4.44.0 pydantic==2.8.0 torch==2.3.1 datasets==2.21.0 seqeval==1.2.2

In [ ]:
import random, json, pathlib, time
import numpy as np
import pandas as pd
import torch
from typing import Optional
from pydantic import BaseModel

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

## Step 1 — Runtime & Paths

In [ ]:
try:
    import google.colab; RUNTIME = 'colab'
    PROJECT_ROOT = pathlib.Path('/content/slm')
except ImportError:
    try:
        import kaggle_secrets; RUNTIME = 'kaggle'
        PROJECT_ROOT = pathlib.Path('/kaggle/working/slm')
    except ImportError:
        RUNTIME = 'local'
        PROJECT_ROOT = pathlib.Path('.').resolve()

DATA_PROC = PROJECT_ROOT / 'data' / 'processed'
REPORTS   = PROJECT_ROOT / 'reports'
MODELS_DIR = PROJECT_ROOT / 'models'
REPORTS.mkdir(parents=True, exist_ok=True)
print(f'Runtime: {RUNTIME}')

## Step 2 — Define Pydantic Schema

`outlines` compiles this into an FSM that masks invalid tokens at each decode step.

In [ ]:
class TicketFields(BaseModel):
    product:    str
    version:    str
    os:         Optional[str] = None
    error_code: Optional[str] = None

# Show the JSON schema outlines will use
import json as _json
schema = TicketFields.model_json_schema()
print('JSON Schema:')
print(_json.dumps(schema, indent=2))

## Step 3 — Load Model for Constrained Decoding

Uses Qwen2.5-1.5B on GPU, or falls back to a smaller model on CPU.

In [ ]:
import outlines

if device == 'cuda':
    MODEL_NAME = 'Qwen/Qwen2.5-1.5B'
else:
    # Smaller fallback for CPU — still demonstrates the pattern
    MODEL_NAME = 'Qwen/Qwen2.5-0.5B'

print(f'Loading {MODEL_NAME} for constrained decoding ...')
try:
    outlines_model = outlines.models.transformers(
        MODEL_NAME,
        model_kwargs={'device_map': 'auto', 'torch_dtype': torch.float16
                      if device == 'cuda' else torch.float32},
    )
    generator = outlines.generate.json(outlines_model, TicketFields)
    print(f'Model loaded. Generator ready.')
    USE_MOCK = False
except Exception as e:
    print(f'Could not load model ({e})')
    print('Using mock generator for demonstration.')
    USE_MOCK = True

## Step 4 — FSM Token Masking Demo

One example showing how constrained decoding produces guaranteed-valid JSON.

In [ ]:
DEMO_TICKETS = [
    ('DeskMate Pro v2.3 crashes with ERR_404 on Windows 10',
     {'product': 'DeskMate Pro', 'version': 'v2.3', 'os': 'Windows 10', 'error_code': 'ERR_404'}),
    ('AgentCore v3.1 freezes on macOS Ventura',
     {'product': 'AgentCore', 'version': 'v3.1', 'os': 'macOS Ventura', 'error_code': None}),
    ('FlowPilot v1.0 slow on Ubuntu 22, error code TIMEOUT',
     {'product': 'FlowPilot', 'version': 'v1.0', 'os': 'Ubuntu 22', 'error_code': 'TIMEOUT'}),
    ('App crashes every login',
     {'product': '', 'version': '', 'os': None, 'error_code': None}),
    ('I cannot use DeskMate v4.0 on Chrome',
     {'product': 'DeskMate', 'version': 'v4.0', 'os': 'Chrome', 'error_code': None}),
]

EXTRACT_PROMPT = (
    'Extract the product name, version number, operating system, and error code '
    'from the support ticket. Output as JSON with keys: product, version, os, error_code. '
    'Use null if a field is not mentioned.\nTicket: '
)

def mock_extract(ticket, gold):
    # Deterministic mock: return gold with 10% chance of one wrong field
    rng = random.Random(hash(ticket) % 2**32)
    result = dict(gold)
    if rng.random() < 0.15 and result['product']:
        result['product'] = result['product'].split()[0]  # truncate multi-word
    return result

print('Demo: constrained extraction on one ticket')
ticket, gold = DEMO_TICKETS[0]
if not USE_MOCK:
    t0     = time.perf_counter()
    result = generator(EXTRACT_PROMPT + ticket)
    ms     = (time.perf_counter() - t0) * 1000
    print(f'Input   : {ticket}')
    print(f'Output  : {result.model_dump()}')
    print(f'Latency : {ms:.0f}ms')
    print(f'Valid JSON: always (guaranteed by FSM)')
else:
    result = mock_extract(ticket, gold)
    print(f'Input   : {ticket}')
    print(f'Output  : {result}')
    print('(mock mode — load model for real constrained decoding)')

## Step 5 — Load Gold Examples with Field Annotations

In [ ]:
# Load annotated examples from Module 2.1 / 2.2
# Fall back to DEMO_TICKETS if gold set not available
gold_path = DATA_PROC / 'gold.jsonl'
field_examples = []

if gold_path.exists():
    df = pd.read_json(gold_path, lines=True)
    if 'fields' in df.columns:
        df = df[df['fields'].apply(
            lambda f: isinstance(f, dict) and len(f) > 0)]
        for _, row in df.head(50).iterrows():
            fields = row['fields']
            field_examples.append({
                'ticket': row['text'],
                'gold': {
                    'product':    fields.get('product', {}).get('value', ''),
                    'version':    fields.get('version', {}).get('value', ''),
                    'os':         fields.get('os', {}).get('value', None),
                    'error_code': fields.get('error_code', {}).get('value', None),
                }
            })

if not field_examples:
    print('No annotated gold examples found — using built-in demo set')
    field_examples = [{'ticket': t, 'gold': g} for t, g in DEMO_TICKETS]
else:
    print(f'Loaded {len(field_examples)} gold examples with field annotations')

## Step 6 — Batch Constrained Extraction

In [ ]:
cd_results = []
cd_latencies = []

for ex in field_examples:
    ticket = ex['ticket']
    t0 = time.perf_counter()
    if not USE_MOCK:
        pred = generator(EXTRACT_PROMPT + ticket).model_dump()
    else:
        pred = mock_extract(ticket, ex['gold'])
    cd_latencies.append((time.perf_counter() - t0) * 1000)
    cd_results.append({'ticket': ticket, 'pred': pred, 'gold': ex['gold']})

print(f'Extracted {len(cd_results)} examples')
print(f'Avg latency: {np.mean(cd_latencies):.0f}ms  '
       f'p95: {np.percentile(cd_latencies, 95):.0f}ms')

# Malformed output check
malformed = sum(
    1 for r in cd_results
    if not isinstance(r['pred'], dict) or 'product' not in r['pred']
)
print(f'Malformed outputs: {malformed}/{len(cd_results)} '
       f'({malformed/len(cd_results)*100:.1f}%) -- should be 0 with constrained decoding')

## Step 7 — Load NER Extractor (Module 2.5)

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

NER_LABELS = ['O','B-product','I-product','B-version','I-version','B-os','I-os']
NER_ID2LABEL = {i: l for i, l in enumerate(NER_LABELS)}

ner_path = MODELS_DIR / 'field_extractor'
if ner_path.exists():
    ner_tokenizer = AutoTokenizer.from_pretrained(str(ner_path))
    ner_model     = AutoModelForTokenClassification.from_pretrained(str(ner_path))
    ner_model.eval()
    print(f'Loaded NER model from {ner_path}')
    USE_NER_MOCK = False
else:
    print('NER model not found (run Module 2.5 first) — using rule-based mock')
    USE_NER_MOCK = True

def extract_ner(ticket):
    if USE_NER_MOCK:
        # Rule-based mock: regex patterns for demo
        import re
        result = {'product': '', 'version': '', 'os': None, 'error_code': None}
        m = re.search(r'(DeskMate Pro|AgentCore|FlowPilot|DeskMate)', ticket)
        if m: result['product'] = m.group(1)
        m = re.search(r'(v[0-9]+\.[0-9]+)', ticket)
        if m: result['version'] = m.group(1)
        m = re.search(r'(Windows 10|macOS [A-Za-z]+|Ubuntu [0-9]+|Chrome)', ticket)
        if m: result['os'] = m.group(1)
        m = re.search(r'(ERR_[0-9A-Z_]+|TIMEOUT|ERROR_[0-9A-Z_]+)', ticket)
        if m: result['error_code'] = m.group(1)
        return result
    else:
        inputs  = ner_tokenizer(ticket, return_tensors='pt', truncation=True, max_length=128)
        with torch.no_grad():
            logits = ner_model(**inputs).logits
        pred_ids = logits[0].argmax(-1).tolist()
        tokens   = ner_tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
        bio_seq  = [NER_ID2LABEL[i] for i in pred_ids]
        result   = {}; cur_field = None; cur_toks = []
        for tok, label in zip(tokens, bio_seq):
            if tok in ('[CLS]','[SEP]','[PAD]'): continue
            if label.startswith('B-'):
                if cur_field:
                    result[cur_field] = ner_tokenizer.convert_tokens_to_string(cur_toks).strip()
                cur_field = label[2:]; cur_toks = [tok]
            elif label.startswith('I-') and cur_field and cur_field == label[2:]:
                cur_toks.append(tok)
            else:
                if cur_field:
                    result[cur_field] = ner_tokenizer.convert_tokens_to_string(cur_toks).strip()
                cur_field = None; cur_toks = []
        if cur_field:
            result[cur_field] = ner_tokenizer.convert_tokens_to_string(cur_toks).strip()
        # Normalise to TicketFields keys
        return {'product': result.get('product',''), 'version': result.get('version',''),
                'os': result.get('os'), 'error_code': result.get('error_code')}

# Test
t = 'DeskMate Pro v2.3 crashes with ERR_404 on Windows 10'
print(f'NER extraction: {extract_ner(t)}')

## Step 8 — Compute Extraction F1 for Both Methods

In [ ]:
def field_f1(preds, golds, field):
    tp = fp = fn = 0
    for pred, gold in zip(preds, golds):
        p_val = (pred.get(field) or '').strip().lower()
        g_val = (gold.get(field) or '').strip().lower()
        if g_val and p_val == g_val: tp += 1
        elif p_val and not g_val:    fp += 1
        elif g_val and not p_val:    fn += 1
        elif g_val and p_val != g_val: fp += 1; fn += 1
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2*precision*recall/(precision+recall) if (precision+recall) > 0 else 0
    return round(f1, 3)

# Run NER on same examples
ner_latencies = []
ner_preds = []
for ex in field_examples:
    t0 = time.perf_counter()
    ner_preds.append(extract_ner(ex['ticket']))
    ner_latencies.append((time.perf_counter() - t0) * 1000)

cd_preds = [r['pred'] for r in cd_results]
golds    = [r['gold'] for r in cd_results]
FIELDS   = ['product', 'version', 'os', 'error_code']

print(f'{"Field":<15} {"CD F1":>8} {"NER F1":>8}')
print('-' * 35)
cd_avg = ner_avg = 0
for field in FIELDS:
    cf = field_f1(cd_preds, golds, field)
    nf = field_f1(ner_preds, golds, field)
    cd_avg += cf; ner_avg += nf
    print(f'{field:<15} {cf:>8.3f} {nf:>8.3f}')
print('-' * 35)
print(f'{"Macro avg":<15} {cd_avg/len(FIELDS):>8.3f} {ner_avg/len(FIELDS):>8.3f}')

## Step 9 — Full Comparison Table

In [ ]:
cd_malformed = malformed
ner_malformed = 0   # NER always outputs valid BIO sequence

cd_mean_ms  = np.mean(cd_latencies)
ner_mean_ms = np.mean(ner_latencies)

cd_f1_avg  = sum(field_f1(cd_preds,  golds, f) for f in FIELDS) / len(FIELDS)
ner_f1_avg = sum(field_f1(ner_preds, golds, f) for f in FIELDS) / len(FIELDS)

print('=== Constrained Decoding vs NER Head ===')
print(f'{"Metric":<30} {"Constrained Dec":>18} {"NER Head (2.5)":>16}')
print('-' * 68)
print(f'{"Extraction macro-F1":<30} {cd_f1_avg:>17.3f} {ner_f1_avg:>15.3f}')
print(f'{"Malformed output rate":<30} {cd_malformed/len(cd_results)*100:>16.1f}% '
       f'{ner_malformed/len(field_examples)*100:>14.1f}%')
print(f'{"Mean latency (ms)":<30} {cd_mean_ms:>17.0f} {ner_mean_ms:>15.0f}')
print(f'{"Schema change cost":<30} {"Edit Pydantic class":>18} {"Retrain model":>16}')
print(f'{"Requires training data":<30} {"No (zero-shot)":>18} {"Yes (BIO spans)":>16}')

## Step 10 — Schema Evolution Demo

Add a new field (`severity`) to the schema and extract it immediately — no retraining.

In [ ]:
class TicketFieldsV2(BaseModel):
    product:    str
    version:    str
    os:         Optional[str] = None
    error_code: Optional[str] = None
    severity:   Optional[str] = None   # NEW: critical / high / medium / low

ticket = 'CRITICAL: DeskMate Pro v2.3 crashes with ERR_500 on Windows 10 blocking all users'

if not USE_MOCK:
    generator_v2 = outlines.generate.json(outlines_model, TicketFieldsV2)
    result_v2    = generator_v2(EXTRACT_PROMPT + ticket)
    print(f'V2 extraction (new severity field):')
    print(result_v2.model_dump())
else:
    print('Schema evolution demo (mock):')
    print({'product': 'DeskMate Pro', 'version': 'v2.3', 'os': 'Windows 10',
           'error_code': 'ERR_500', 'severity': 'critical'})
print()
print('NER head would need: label new BIO spans + retrain. Constrained decoding: zero cost.')

## Step 11 — Write Benchmark Report

In [ ]:
cd_wins_f1    = cd_f1_avg >= ner_f1_avg
ner_wins_lat  = ner_mean_ms < cd_mean_ms

if ner_wins_lat and not cd_wins_f1:
    recommendation = (
        'Use NER head (Module 2.5) for production. '
        'Higher F1 and substantially lower latency. '
        'Switch to constrained decoding if the extraction schema changes frequently.'
    )
elif cd_wins_f1 and not ner_wins_lat:
    recommendation = (
        'Consider constrained decoding if schema evolution is expected. '
        'NER head is faster; constrained decoding scores higher F1 but at significant latency cost.'
    )
else:
    recommendation = (
        'Both methods are comparable on F1. '
        'Prefer NER head for latency-sensitive production. '
        'Prefer constrained decoding for zero-shot flexibility and schema evolution.'
    )

lines = [
    '# Constrained Decoding vs NER Head Benchmark\n',
    f'| Metric | Constrained Decoding | NER Head (2.5) |',
    f'|---|---|---|',
    f'| Extraction macro-F1 | {cd_f1_avg:.3f} | {ner_f1_avg:.3f} |',
    f'| Malformed output rate | {cd_malformed/len(cd_results)*100:.1f}% | 0.0% |',
    f'| Mean latency | {cd_mean_ms:.0f}ms | {ner_mean_ms:.0f}ms |',
    f'| Requires training data | No | Yes |',
    f'| Schema change cost | Edit Pydantic class | Retrain |',
    '',
    f'## Recommendation\n',
    recommendation,
]
report_path = REPORTS / 'constrained_decoding_benchmark.md'
report_path.write_text('\n'.join(lines))
print(f'Report written: {report_path}')
print()
print(recommendation)

## Checkpoint

> **Constrained decoding guarantees a valid JSON structure. What does it NOT guarantee, and when would you still prefer a fine-tuned NER head?**

Write your answer below.

In [ ]:
answer = """
[Write your answer here]
"""
print(answer)

## Deliverable Summary

| Artifact | Location |
|---|---|
| Benchmark report | `reports/constrained_decoding_benchmark.md` |

**What you've built:** a JSON-schema-constrained extractor with zero malformed-output rate,
benchmarked against the fine-tuned NER head from Module 2.5.

**Next:** Module 3.3 — PEFT theory: LoRA and QLoRA.